In [1]:
# 1. MovieLens 1M 데이터셋 다운로드
!wget -q https://files.grouplens.org/datasets/movielens/ml-1m.zip

# 2. 압축 해제
!unzip -o ml-1m.zip

# 3. 파일 확인 (ratings.dat이 있으면 성공)
!ls -lh ml-1m/

Archive:  ml-1m.zip
   creating: ml-1m/
  inflating: ml-1m/movies.dat        
  inflating: ml-1m/ratings.dat       
  inflating: ml-1m/README            
  inflating: ml-1m/users.dat         
total 24M
-rw-r----- 1 root root 168K Mar 26  2003 movies.dat
-rw-r----- 1 root root  24M Feb 28  2003 ratings.dat
-rw-r----- 1 root root 5.5K Jan 29  2016 README
-rw-r----- 1 root root 132K Feb 28  2003 users.dat


In [2]:
import hashlib
import sys
import time

# 데이터 스트림 제너레이터 (코랩에 다운로드된 경로 적용)
def stream_data(file_path="ml-1m/ratings.dat"):
    with open(file_path, 'r', encoding='latin-1') as f:
        for line in f:
            parts = line.strip().split("::")
            if len(parts) >= 2:
                yield parts[1] # MovieID 추출

# 1. Bloom Filter 구현
class BloomFilter:
    def __init__(self, size, num_hashes):
        self.size = size
        self.num_hashes = num_hashes
        self.bit_array = [0] * size

    def _hashes(self, item):
        hashes = []
        for i in range(self.num_hashes):
            h = int(hashlib.md5((str(item) + str(i)).encode('utf-8')).hexdigest(), 16)
            hashes.append(h % self.size)
        return hashes

    def add(self, item):
        for position in self._hashes(item):
            self.bit_array[position] = 1

    def check(self, item):
        for position in self._hashes(item):
            if self.bit_array[position] == 0:
                return False
        return True

    def get_memory_bytes(self):
        return sys.getsizeof(self.bit_array)

# 2. Count-Min Sketch 구현
class CountMinSketch:
    def __init__(self, width, depth):
        self.width = width
        self.depth = depth
        self.table = [[0] * width for _ in range(depth)]

    def _hashes(self, item):
        hashes = []
        for i in range(self.depth):
            h = int(hashlib.md5((str(item) + str(i)).encode('utf-8')).hexdigest(), 16)
            hashes.append(h % self.width)
        return hashes

    def add(self, item):
        for i, position in enumerate(self._hashes(item)):
            self.table[i][position] += 1

    def estimate(self, item):
        return min(self.table[i][position] for i, position in enumerate(self._hashes(item)))

    def get_memory_bytes(self):
        return sys.getsizeof(self.table) + sum(sys.getsizeof(row) for row in self.table)

In [3]:
# 파일 경로 설정
file_path = "ml-1m/ratings.dat"

print("="*60)
print("[STEP 03] Ground Truth 계산 중...")
print("="*60)
gt_set = set()
gt_dict = {}

t_start = time.time()
for movie in stream_data(file_path):
    gt_set.add(movie)
    gt_dict[movie] = gt_dict.get(movie, 0) + 1
gt_time = time.time() - t_start

print(f"✔ Ground Truth 완료 (시간: {gt_time:.2f}초)")
print(f"✔ 총 레코드 수: {sum(gt_dict.values()):,}개 | 고유 영화 수: {len(gt_set):,}개\n")

# ---------------------------------------------------------
# Bloom Filter 실험
# ---------------------------------------------------------
print("="*60)
print("[실험 1] Bloom Filter 파라미터(Size) 비교")
print("="*60)
test_non_exist = [str(i) for i in range(90000, 90100)] # 없는 데이터로 오탐률 테스트

for size in [10000, 50000]:
    bf = BloomFilter(size=size, num_hashes=3)

    t_start = time.time()
    for movie in stream_data(file_path):
        bf.add(movie)
    bf_time = time.time() - t_start

    fp_count = sum(1 for m in test_non_exist if bf.check(m))
    fpr = (fp_count / len(test_non_exist)) * 100

    print(f"▶ Bit Array Size: {size:5d} | 시간: {bf_time:.2f}초 | 메모리: {bf.get_memory_bytes():6d} bytes | 오탐률(FPR): {fpr:.1f}%")

# ---------------------------------------------------------
# Count-Min Sketch 실험
# ---------------------------------------------------------
print("\n" + "="*60)
print("[실험 2] Count-Min Sketch 파라미터(Width) 비교")
print("="*60)

for width in [500, 2000]:
    cms = CountMinSketch(width=width, depth=3)

    t_start = time.time()
    for movie in stream_data(file_path):
        cms.add(movie)
    cms_time = time.time() - t_start

    mae = 0
    for m in gt_dict.keys():
        mae += abs(cms.estimate(m) - gt_dict[m])
    mae /= len(gt_dict)

    print(f"▶ Table Width: {width:4d} | 시간: {cms_time:.2f}초 | 메모리: {cms.get_memory_bytes():6d} bytes | 평균절대오차(MAE): {mae:.2f}")

[STEP 03] Ground Truth 계산 중...
✔ Ground Truth 완료 (시간: 0.57초)
✔ 총 레코드 수: 1,000,209개 | 고유 영화 수: 3,706개

[실험 1] Bloom Filter 파라미터(Size) 비교
▶ Bit Array Size: 10000 | 시간: 4.28초 | 메모리:  80056 bytes | 오탐률(FPR): 32.0%
▶ Bit Array Size: 50000 | 시간: 3.77초 | 메모리: 400056 bytes | 오탐률(FPR): 1.0%

[실험 2] Count-Min Sketch 파라미터(Width) 비교
▶ Table Width:  500 | 시간: 4.38초 | 메모리:  12256 bytes | 평균절대오차(MAE): 1026.51
▶ Table Width: 2000 | 시간: 4.21초 | 메모리:  48256 bytes | 평균절대오차(MAE): 96.50
